<a href="https://colab.research.google.com/github/anmol6027/ai-native-engineering/blob/main/03-foundation-models-token-economics/model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 03 — Model Selection Under Constraints

**Build Task 03.2**

### The question

A company classifies **50,000 support tickets a day**. Three models can do the job. Which one do you ship?

### Why this matters

| | |
|---|---|
| **Cost** | what you pay per call, and at volume |
| **Capability** | does it get the answer right |
| **Risk** | does it get the *same* answer every time |

Everyone compares **capability** — benchmark scores are all anyone talks about.
Almost nobody measures the other two **on their own task**.

This build measures all three.

### What you'll produce

A comparison table you could put in front of a CTO.

---

## Cell 1 — Install

In [ ]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.4 MB/s eta 0:00:00


## Cell 2 — Setup

Same Groq API key from Topic 01 — already in your Colab secrets as `GROQ_API_KEY`.

If you get a `SecretNotFoundError`: click the 🔑 icon in the left sidebar, check the secret exists, and make sure **Notebook access** is toggled on.

In [ ]:
from groq import Groq
from google.colab import userdata
import time

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print("Connected")

Connected


## Cell 3 — Which models do you actually have?

Groq rotates models frequently. **Don't assume — check.**

This is itself a lesson worth noting: in Topic 01 the model I gave you had been retired and returned a 404. The model you build on today can disappear tomorrow. That's vendor risk, and it's one of the four risk categories from the lecture.

In [ ]:
models = client.models.list()

print("AVAILABLE MODELS")
print("-" * 50)
for m in models.data:
    print(" ", m.id)

AVAILABLE MODELS
--------------------------------------------------
  whisper-large-v3-turbo
  openai/gpt-oss-20b
  openai/gpt-oss-120b
  groq/compound-mini
  meta-llama/llama-prompt-guard-2-86m
  canopylabs/orpheus-v1-english
  qwen/qwen3.8-27b
  whisper-large-v3
  allam-2-7b
  openai/gpt-oss-safeguard-20b
  canopylabs/orpheus-arabic-saudi
  qwen/qwen3.6-27b
  groq/compound
  meta-llama/llama-prompt-guard-2-22m


## Cell 4 — Pick three

**This is the one cell you need to edit.**

Choose three **chat** models from the list Cell 3 printed. Skip anything for:
- audio (`whisper`)
- speech (`orpheus`)
- safety filtering (`prompt-guard`)

Those can't classify text.

**Pick deliberately: one large, one small, one middle-sized.**

The whole point is to find out whether the expensive one earns its price *on this specific task*.

In [ ]:
# EDIT THESE to match what Cell 3 printed

MODELS = [
    "openai/gpt-oss-120b",     # large
    "openai/gpt-oss-20b",      # small
    "qwen/qwen3.8-27b",        # middle
]

print("Comparing:")
for m in MODELS:
    print(" ", m)

Comparing:
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  qwen/qwen3.8-27b


## Cell 5 — The ticket

Same ambiguous ticket from Topic 01 — reused deliberately. A fair comparison needs identical input.

It contains **three problems at once**: an app crash, a double charge, and a refund request. That ambiguity is what makes it a real test.

In [ ]:
TICKET = """
Hi, I've been trying to place an order since yesterday.
The app froze twice during payment. Both times my bank sent
an SMS that money was deducted, but I never got an order
confirmation email. Now I can see two charges on my card
but nothing in my order history. I've been shopping with
you for 3 years and this has never happened. I just want
my money back. Also the app is still crashing when I open
the checkout page.
"""

LABELS = ["order_issue", "refund_request", "app_bug",
          "payment_issue", "account"]

print("Ticket loaded.")

Ticket loaded.


## Cell 6 — The classifier

One function, works with any model.

We also **time each call** — because latency is the third thing nobody measures. A model that's cheap and stable but takes four seconds is unusable in a live support flow.

In [ ]:
def classify(model, ticket, temperature=0.2):
    """
    Returns the label, token counts, and how long it took.
    """
    prompt = f"""Classify this customer support ticket into exactly ONE intent.

Available intents: {', '.join(LABELS)}

Ticket: {ticket}

Reply with only the intent label. Nothing else."""

    # Start the stopwatch BEFORE the call
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=300
    )

    # Stop it after. Difference = latency in seconds.
    elapsed = time.time() - start

    return {
        "label": response.choices[0].message.content.strip(),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "latency": elapsed
    }


# Smoke test before running the full comparison
test = classify(MODELS[0], TICKET)
print("Test run OK")
print("  Label  :", test["label"])
print("  Latency:", round(test["latency"], 2), "seconds")

Test run OK
  Label  : refund_request
  Latency: 0.64 seconds


## Cell 7 — Run the comparison

Each model classifies the **same ticket five times**.

**Why five?** One run tells you if a model *can* answer. Five runs tell you if it answers the *same way*.

Those are different questions, and only the second one matters in production. If the same ticket routes three different ways, customers get inconsistent treatment and nobody knows why.

This takes 30–60 seconds to run.

In [ ]:
RUNS = 5
results = {}

for model in MODELS:
    print(f"\nTesting {model}")
    print("-" * 50)

    labels = []
    latencies = []
    input_tokens = 0
    output_tokens = 0

    for i in range(RUNS):
        try:
            r = classify(model, TICKET)
            labels.append(r["label"])
            latencies.append(r["latency"])
            input_tokens += r["input_tokens"]
            output_tokens += r["output_tokens"]
            print(f"  Run {i+1}: {r['label']:<18} {r['latency']:.2f}s")
        except Exception as e:
            print(f"  Run {i+1}: FAILED — {e}")

    if labels:
        results[model] = {
            "labels": labels,
            "unique": len(set(labels)),
            "avg_input": input_tokens / len(labels),
            "avg_output": output_tokens / len(labels),
            "avg_latency": sum(latencies) / len(latencies)
        }

print("\nDone.")


Testing openai/gpt-oss-120b
--------------------------------------------------
  Run 1: refund_request     0.35s
  Run 2: refund_request     0.52s
  Run 3: refund_request     0.37s
  Run 4: refund_request     0.60s
  Run 5: refund_request     0.35s

Testing openai/gpt-oss-20b
--------------------------------------------------
  Run 1:                    0.78s
  Run 2: refund_request     0.64s
  Run 3: refund_request     0.46s
  Run 4: refund_request     0.36s
  Run 5: refund_request     0.42s

Testing qwen/qwen3.8-27b
--------------------------------------------------
  Run 1: payment_issue      0.13s
  Run 2: payment_issue      0.14s
  Run 3: payment_issue      0.13s
  Run 4: payment_issue      0.13s
  Run 5: payment_issue      0.12s

Done.


## Cell 8 — The comparison table

Turning raw numbers into a decision.

**On pricing:** Groq's rates differ per model. The figures below are approximate — check `console.groq.com` for current numbers and update the dictionary if you want exact costs.

If you picked different models in Cell 4, add them to `PRICING` too.

In [ ]:
PRICING = {
    "openai/gpt-oss-120b": {"in": 0.59, "out": 0.79},
    "openai/gpt-oss-20b":  {"in": 0.10, "out": 0.50},
    "qwen/qwen3.8-27b":    {"in": 0.29, "out": 0.59},
}

USD_TO_INR = 88
TICKETS_PER_DAY = 50_000
DAYS = 30


def monthly_cost_inr(model, avg_in, avg_out):
    rates = PRICING.get(model, {"in": 0.50, "out": 0.75})
    per_call = (avg_in / 1_000_000 * rates["in"]) + \
               (avg_out / 1_000_000 * rates["out"])
    return per_call * TICKETS_PER_DAY * DAYS * USD_TO_INR


print("MODEL COMPARISON")
print("=" * 76)
print(f"{'Model':<26} {'Answer':<16} {'Stable':<8} {'Latency':<10} {'Rs/month':<12}")
print("-" * 76)

for model, r in results.items():
    answer = max(set(r["labels"]), key=r["labels"].count)
    stable = "yes" if r["unique"] == 1 else f"no ({r['unique']})"
    cost = monthly_cost_inr(model, r["avg_input"], r["avg_output"])

    short_name = model.split("/")[-1][:24]
    print(f"{short_name:<26} {answer:<16} {stable:<8} "
          f"{r['avg_latency']:.2f}s{'':<5} Rs {cost:>9,.0f}")

print("=" * 76)
print(f"At {TICKETS_PER_DAY:,} tickets/day over {DAYS} days.")

MODEL COMPARISON
Model                      Answer           Stable   Latency    Rs/month    
----------------------------------------------------------------------------
gpt-oss-120b               refund_request   yes      0.44s      Rs    26,184
gpt-oss-20b                refund_request   no (2)   0.53s      Rs    17,899
qwen3.8-27b                payment_issue    yes      0.13s      Rs     6,167
At 50,000 tickets/day over 30 days.


---

## What to look for

Three possible outcomes, and **all of them are good findings**:

**The small model matched the large one exactly.**
You've found real money. The premium bought nothing on this task.

**The small model flipped its answer, the large one held.**
Now you know what stability costs — and you can defend paying for it.

**A model was cheap and stable but too slow.**
Latency is the constraint nobody budgets for. That's a finding too.

You're not hoping for a specific result. You're producing the comparison almost nobody bothers to run.